# Kurinda: Feature Engineering

**Notebook 02 of 03** ·

Builds the **sector-month master dataset** that feeds the LightGBM model.

**Inputs**:
- `data/processed/dhs/dhs_cluster_2019_20.csv` (500 DHS clusters, from notebook 01)
- `data/raw/dhs/2019-20/RWGE81FL.zip` (DHS displaced GPS shapefile)
- `data/raw/geo/gadm41_RWA.gpkg` (422 GADM sectors)
- `data/raw/chirps/chirps_sector_monthly.csv` (30,384 sector-months)
- `data/raw/ndvi/ndvi_sector_monthly.csv` (30,384 sector-months)
- `data/raw/wfp/wfp_district_monthly.csv` (2,253 district-month-commodity rows)

**Output**: `data/processed/master/sector_month_features.parquet`
(~30,384 rows × ~30 features, ready for modeling)

## 0. Imports and configuration

Same `.venv` as notebook 01. Adds `numpy` for vectorized operations and the `warnings` filter to suppress GeoPandas's deprecation chatter that doesn't affect our pipeline.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="geopandas")

from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

# Paths (all relative to ml/notebooks/)
DHS_CLUSTER  = Path("../../data/processed/dhs/dhs_cluster_2019_20.csv")
DHS_GPS_ZIP  = Path("../../data/raw/dhs/2019-20/RWGE81FL.zip")
GADM_GPKG    = Path("../../data/raw/geo/gadm41_RWA.gpkg")
CHIRPS_CSV   = Path("../../data/raw/chirps/chirps_sector_monthly.csv")
NDVI_CSV     = Path("../../data/raw/ndvi/ndvi_sector_monthly.csv")
WFP_CSV      = Path("../../data/raw/wfp/wfp_district_monthly.csv")

# DHS sector aggregate (produced by Cell 22; consumed by Cell 31)
DHS_SECTOR   = Path("../../data/processed/dhs/dhs_sector_2019_20.csv")

# Master dataset output
MASTER_OUT   = Path("../../data/processed/master/sector_month_features.parquet")

# Verify all inputs exist before doing anything else
for p in [DHS_CLUSTER, DHS_GPS_ZIP, GADM_GPKG, CHIRPS_CSV, NDVI_CSV, WFP_CSV]:
    status = "OK     " if p.exists() else "MISSING"
    print(f"{status}  {p}")

print(f"\nOutput will be written to: {MASTER_OUT}")

OK       ..\..\data\processed\dhs\dhs_cluster_2019_20.csv
OK       ..\..\data\raw\dhs\2019-20\RWGE81FL.zip
OK       ..\..\data\raw\geo\gadm41_RWA.gpkg
OK       ..\..\data\raw\chirps\chirps_sector_monthly.csv
OK       ..\..\data\raw\ndvi\ndvi_sector_monthly.csv
OK       ..\..\data\raw\wfp\wfp_district_monthly.csv

Output will be written to: ..\..\data\processed\master\sector_month_features.parquet


In [2]:
import sys
print(sys.executable)

c:\Users\Hello\kurinda\backend\.venv\Scripts\python.exe


## 1. Spatial join — DHS clusters → GADM sectors

DHS surveys at the **cluster** level (~30 households per cluster, 500 clusters in Rwanda 2019-20). GADM provides 422 sector polygons. To use DHS as a sector-level feature, we need a cluster → sector mapping.

**Privacy displacement**: DHS deliberately displaces GPS coordinates (urban ≤2 km, rural ≤5 km, 1% rural ≤10 km). Rwandan sectors average ~56 km² (~4–8 km radius), so the displacement is smaller than the typical sector. The vast majority of displaced points still land in their true sector. Edge cases (clusters near sector boundaries) are handled via a nearest-sector fallback.

Four sub-steps follow:

1. Load DHS aggregate, GPS shapefile, GADM sectors → verify cluster ID alignment
2. Merge DHS aggregate with GPS, then spatial join (point-in-polygon) against sectors
3. Nearest-sector fallback for any clusters that fell outside all polygons
4. Aggregate clusters → sectors weighted by `n_children`, save

### 1a. Load all three sources

In [3]:
# DHS cluster aggregate (output of notebook 01)
dhs = pd.read_csv(DHS_CLUSTER)
print(f"DHS cluster aggregate: {len(dhs):,} rows × {len(dhs.columns)} cols")
print(dhs.head(3).to_string(index=False))

DHS cluster aggregate: 500 rows × 9 cols
 cluster_id  n_children  stunting_rate  severe_rate  mean_haz  region  urban_rural  mean_wealth_q  mean_maternal_edu
          1         6.0      33.333333          0.0 -1.431667     1.0          2.0       4.666667           9.833333
          2         5.0      20.000000          0.0 -0.910000     2.0          2.0       1.600000           4.200000
          3         5.0      40.000000         20.0 -1.806000     4.0          2.0       2.600000           4.400000


In [4]:
# DHS GPS shapefile (read directly from inside the zip)
gps = gpd.read_file(DHS_GPS_ZIP)
print(f"GPS points: {len(gps):,}")
print(f"CRS: {gps.crs}")
print()
print(gps[["DHSCLUST", "URBAN_RURA", "LATNUM", "LONGNUM"]].head(3).to_string(index=False))
print()
print(f"Urban/rural split:")
print(gps["URBAN_RURA"].value_counts())

GPS points: 500
CRS: EPSG:4326

 DHSCLUST URBAN_RURA    LATNUM   LONGNUM
      1.0          R -1.905014 30.040970
      2.0          R -2.761771 29.844323
      3.0          R -1.682106 29.855809

Urban/rural split:
URBAN_RURA
R    388
U    112
Name: count, dtype: int64


In [5]:
# GADM sectors (ADM_ADM_3 layer)
sectors = gpd.read_file(GADM_GPKG, layer="ADM_ADM_3")
print(f"GADM sectors: {len(sectors):,}")
print(f"CRS: {sectors.crs}")
print()
print(sectors[["GID_3", "NAME_3", "NAME_2", "NAME_1"]].head(3).to_string(index=False))
print()
print(f"CRS aligned with GPS: {'✅' if gps.crs == sectors.crs else '❌'}")

GADM sectors: 422
CRS: EPSG:4326

      GID_3  NAME_3 NAME_2       NAME_1
RWA.1.1.1_1  Bungwe Burera Amajyaruguru
RWA.1.1.2_1  Butaro Burera Amajyaruguru
RWA.1.1.3_1 Cyanika Burera Amajyaruguru

CRS aligned with GPS: ✅


### 1b. Verify cluster ID alignment between DHS aggregate and GPS file

DHS uses `cluster_id` in the data file and `DHSCLUST` in the GPS file. They should be identical sets every cluster in one must appear in the other. The symmetric difference (`^`) catches mismatches in either direction.

In [6]:
dhs_ids = set(dhs["cluster_id"].astype(int))
gps_ids = set(gps["DHSCLUST"].astype(int))

print(f"DHS aggregate IDs: {min(dhs_ids)}..{max(dhs_ids)} ({len(dhs_ids)} unique)")
print(f"GPS file IDs:      {min(gps_ids)}..{max(gps_ids)} ({len(gps_ids)} unique)")

mismatch = dhs_ids ^ gps_ids
if mismatch:
    print(f"Mismatch ({len(mismatch)} IDs): {sorted(mismatch)[:10]}...")
else:
    print(f"Cluster IDs match perfectly across both files.")

DHS aggregate IDs: 1..500 (500 unique)
GPS file IDs:      1..500 (500 unique)
Cluster IDs match perfectly across both files.


### 1c. Drop clusters with missing GPS, then merge DHS + GPS

DHS encodes missing GPS as `LATNUM = 0, LONGNUM = 0`. If we leave these in, the spatial join silently places them at the equator/prime meridian, then the nearest-sector fallback "rescues" them to Rwanda's south-west corner producing nonsense. Filter them out before joining.

In [7]:
# Drop clusters with missing GPS (LATNUM == 0 is DHS's missing-coordinate code)
n_before = len(gps)
gps = gps[gps["LATNUM"] != 0].copy()
n_dropped = n_before - len(gps)
print(f"Dropped {n_dropped} clusters with missing GPS ({n_before} → {len(gps)})")

# Prepare GPS for merge: rename DHSCLUST → cluster_id to match dhs frame
gps_minimal = gps[["DHSCLUST", "URBAN_RURA", "LATNUM", "LONGNUM", "geometry"]].copy()
gps_minimal["cluster_id"] = gps_minimal["DHSCLUST"].astype(int)
gps_minimal = gps_minimal.drop(columns=["DHSCLUST"])

# Merge DHS aggregate (stunting rates + covariates) with GPS (geometry)
dhs_with_geo = gps_minimal.merge(dhs, on="cluster_id", how="inner")
print(f"\nAfter merge: {len(dhs_with_geo):,} rows (expect ~500 minus dropped)")
print(dhs_with_geo[["cluster_id", "URBAN_RURA", "stunting_rate", "n_children"]].head(3).to_string(index=False))

Dropped 0 clusters with missing GPS (500 → 500)

After merge: 500 rows (expect ~500 minus dropped)
 cluster_id URBAN_RURA  stunting_rate  n_children
          1          R      33.333333         6.0
          2          R      20.000000         5.0
          3          R      40.000000         5.0


### 2. Spatial join — point-in-polygon

For each cluster's displaced GPS point, find the sector polygon that contains it. `predicate="within"` means strict containment a point on a sector boundary still counts as inside that sector.

Most clusters will match. A handful near national borders may fall outside all polygons due to displacement those go through the nearest-sector fallback in the next step.

In [8]:
joined = gpd.sjoin(
    dhs_with_geo,
    sectors[["GID_3", "NAME_3", "NAME_2", "NAME_1", "geometry"]],
    how="left",
    predicate="within",
)

matched = joined["GID_3"].notna().sum()
unmatched = len(joined) - matched
match_rate = matched / len(joined) * 100

print(f"Within-polygon matches: {matched} / {len(joined)} ({match_rate:.1f}%)")
print(f"Unmatched (fall through to nearest-sector): {unmatched}")

if unmatched > 0:
    print(f"\nUnmatched clusters:")
    print(joined[joined["GID_3"].isna()][
        ["cluster_id", "URBAN_RURA", "LATNUM", "LONGNUM"]
    ].to_string(index=False))

Within-polygon matches: 498 / 500 (99.6%)
Unmatched (fall through to nearest-sector): 2

Unmatched clusters:
 cluster_id URBAN_RURA    LATNUM   LONGNUM
        194          R -1.325257 29.841897
        396          R -1.357164 29.778110


### 3. Nearest-sector fallback for unmatched clusters

For clusters that fell outside all sector polygons (typically near national borders due to GPS displacement), assign each one to its **nearest** sector via `sjoin_nearest`. Ensures zero DHS data loss.

In [9]:
if unmatched > 0:
    unmatched_mask = joined["GID_3"].isna()
    unmatched_points = (
        joined[unmatched_mask]
        .drop(columns=["GID_3", "NAME_3", "NAME_2", "NAME_1", "index_right"])
    )

    nearest = gpd.sjoin_nearest(
        unmatched_points,
        sectors[["GID_3", "NAME_3", "NAME_2", "NAME_1", "geometry"]],
        how="left",
        distance_col="dist_to_nearest_m",
    )

    for col in ["GID_3", "NAME_3", "NAME_2", "NAME_1"]:
        joined.loc[unmatched_mask, col] = nearest[col].values

    print(f"Recovered {unmatched} cluster(s) via nearest-sector fallback:")
    print(joined[unmatched_mask][
        ["cluster_id", "URBAN_RURA", "GID_3", "NAME_3", "NAME_2"]
    ].to_string(index=False))
else:
    print("No fallback needed — all clusters matched within a sector.")

# Final assertion: every cluster must now have a sector
assert joined["GID_3"].notna().all(), "Some clusters still unassigned!"

sectors_covered = joined["GID_3"].nunique()
print(f"\n All {len(joined)} clusters assigned to sectors.")
print(f" Sectors with DHS data: {sectors_covered} / {len(sectors)} ({100*sectors_covered/len(sectors):.1f}%)")
print(f" Sectors WITHOUT DHS data: {len(sectors) - sectors_covered}")

Recovered 2 cluster(s) via nearest-sector fallback:
 cluster_id URBAN_RURA        GID_3    NAME_3 NAME_2
        194          R RWA.1.1.10_1 Kinyababa Burera
        396          R  RWA.1.1.8_1    Kagogo Burera

 All 500 clusters assigned to sectors.
 Sectors with DHS data: 320 / 422 (75.8%)
 Sectors WITHOUT DHS data: 102


### 4. Aggregate clusters → sectors (weighted by n_children)

Multiple clusters often fall within the same sector (especially urban). Combine them using sample-size weights so a cluster with 30 children counts more than a cluster with 5.

Variables aggregated per sector:
- `stunting_rate`, `severe_rate`, `mean_haz`: weighted means
- `mean_wealth_q`, `mean_maternal_edu`: weighted means
- `n_clusters`: how many clusters contributed (data confidence indicator)
- `n_children`: total sample size in the sector

In [10]:
def wmean(values, weights):
    """Weighted mean, ignoring NaN values and zero weights."""
    mask = values.notna() & weights.notna() & (weights > 0)
    if mask.sum() == 0:
        return np.nan
    return (values[mask] * weights[mask]).sum() / weights[mask].sum()

sector_agg = (
    joined.groupby(["GID_3", "NAME_3", "NAME_2", "NAME_1"])
          .apply(lambda g: pd.Series({
              "n_clusters":        len(g),
              "n_children":        g["n_children"].sum(),
              "stunting_rate":     wmean(g["stunting_rate"],     g["n_children"]),
              "severe_rate":       wmean(g["severe_rate"],       g["n_children"]),
              "mean_haz":          wmean(g["mean_haz"],          g["n_children"]),
              "mean_wealth_q":     wmean(g["mean_wealth_q"],     g["n_children"]),
              "mean_maternal_edu": wmean(g["mean_maternal_edu"], g["n_children"]),
          }), include_groups=False)
          .reset_index()
)

print(f"Sector-level aggregate: {len(sector_agg)} sectors")
print()
print("First 5 rows:")
print(sector_agg.head().to_string(index=False))
print()
print("Stunting rate distribution across sectors:")
print(sector_agg["stunting_rate"].describe().round(1))
print()
print("Clusters-per-sector distribution:")
print(sector_agg["n_clusters"].value_counts().sort_index())

Sector-level aggregate: 320 sectors

First 5 rows:
       GID_3    NAME_3 NAME_2       NAME_1  n_clusters  n_children  stunting_rate  severe_rate  mean_haz  mean_wealth_q  mean_maternal_edu
RWA.1.1.10_1 Kinyababa Burera Amajyaruguru         3.0        28.0      57.142857    21.428571 -2.189643       2.214286           4.357143
RWA.1.1.12_1     Nemba Burera Amajyaruguru         1.0         6.0      16.666667    16.666667 -1.355000       3.833333          10.166667
 RWA.1.1.2_1    Butaro Burera Amajyaruguru         1.0         7.0      28.571429     0.000000 -1.771429       4.285714           7.428571
 RWA.1.1.3_1   Cyanika Burera Amajyaruguru         2.0        19.0      36.842105    10.526316 -1.779474       2.578947           4.368421
 RWA.1.1.4_1     Cyeru Burera Amajyaruguru         2.0        16.0      50.000000    18.750000 -1.961250       1.750000           5.125000

Stunting rate distribution across sectors:
count    320.0
mean      33.0
std       18.6
min        0.0
25%       1

### Sanity check: weighted national stunting rate from sector aggregate

Computing the national stunting rate from the **sector aggregate** (weighted by total children) should reproduce the 33% from notebook 01. If it doesn't, our weighting is wrong somewhere.

In [11]:
total_children = sector_agg["n_children"].sum()
national_stunting = wmean(sector_agg["stunting_rate"], sector_agg["n_children"])

print(f"Children represented in sector aggregate: {total_children:,.0f}")
print(f"National stunting rate from sectors:      {national_stunting:.1f}%")
print(f"From notebook 01 (child-level weighted):  33.0%")
print(f"Published Rwanda DHS 2019-20:             ~33.0%")

if abs(national_stunting - 33.0) < 1.5:
    print("\n Sector aggregation preserves the national stunting rate.")
else:
    print(f"\n Discrepancy of {abs(national_stunting - 33.0):.1f}pp — investigate before continuing.")

Children represented in sector aggregate: 3,809
National stunting rate from sectors:      33.4%
From notebook 01 (child-level weighted):  33.0%
Published Rwanda DHS 2019-20:             ~33.0%

 Sector aggregation preserves the national stunting rate.


### Save sector aggregate

Persist `dhs_sector_2019_20.csv` so the master dataset build (next section) can read it directly.

In [12]:
out_path = Path("../../data/processed/dhs/dhs_sector_2019_20.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
sector_agg.to_csv(out_path, index=False)

print(f" Saved: {out_path.resolve()}")
print(f" {len(sector_agg)} rows × {len(sector_agg.columns)} columns")
print(f" File size: {out_path.stat().st_size / 1024:.1f} KB")

 Saved: C:\Users\Hello\kurinda\data\processed\dhs\dhs_sector_2019_20.csv
 320 rows × 11 columns
 File size: 37.7 KB


## 2. Build the sector-month master dataset

We now have five data sources at three different grains:

| Source | Grain | Rows |
|---|---|---|
| DHS stunting | Sector (320) | 320 |
| CHIRPS rainfall | Sector × month | 30,384 |
| MODIS NDVI | Sector × month | 30,384 |
| WFP prices | District × month × commodity | 2,253 |
| GADM sectors | Sector | 422 |

**Goal**: produce one dataframe where each row is a `(GID_3, year, month)` and every feature is a column.

**Target shape**: 422 sectors × 72 months = **30,384 rows**.

### Three modeling decisions baked into the build

1. **DHS coverage gap**: 102 sectors have no DHS data → `stunting_rate = NaN`. Model trains on labeled subset (320), predicts on all 422.
2. **DHS is time-static**: 2019-20 stunting values broadcast across all 72 months. Captures sector's baseline vulnerability.
3. **WFP cascade fallback**: district price → province mean → national mean, so every sector-month has a price signal.

### 2a. Build the sector-month skeleton

Cartesian product of all 422 sectors × 72 months. Every downstream join lands on this scaffold, so we never lose rows accidentally.

In [13]:
# All 422 sectors from GADM (one row per sector with admin metadata)
sectors_meta = (
    sectors[["GID_3", "NAME_3", "NAME_2", "NAME_1"]]
    .drop_duplicates(subset=["GID_3"])
    .reset_index(drop=True)
)

# All 72 months (2020-01 to 2025-12)
months = pd.DataFrame(
    [(y, m) for y in range(2020, 2026) for m in range(1, 13)],
    columns=["year", "month"]
)

# Cartesian product: 422 × 72 = 30,384 rows
master = sectors_meta.merge(months, how="cross")

print(f"Master skeleton: {len(master):,} rows")
print(f"  = {sectors_meta.shape[0]} sectors × {len(months)} months")
print()
print("First 5 rows:")
print(master.head().to_string(index=False))

Master skeleton: 30,384 rows
  = 422 sectors × 72 months

First 5 rows:
      GID_3 NAME_3 NAME_2       NAME_1  year  month
RWA.1.1.1_1 Bungwe Burera Amajyaruguru  2020      1
RWA.1.1.1_1 Bungwe Burera Amajyaruguru  2020      2
RWA.1.1.1_1 Bungwe Burera Amajyaruguru  2020      3
RWA.1.1.1_1 Bungwe Burera Amajyaruguru  2020      4
RWA.1.1.1_1 Bungwe Burera Amajyaruguru  2020      5


### 2b. Attach CHIRPS rainfall

Left-join CHIRPS on `(GID_3, year, month)`. Every sector-month should match exactly one CHIRPS row.

In [14]:
chirps = pd.read_csv(CHIRPS_CSV)
print(f"CHIRPS: {len(chirps):,} rows × {len(chirps.columns)} cols")
print(f"Columns: {list(chirps.columns)}")

# Keep only the join keys + value (drop redundant name columns)
chirps_slim = chirps[["GID_3", "year", "month", "rainfall_mm"]]

# Left-join onto master
master = master.merge(chirps_slim, on=["GID_3", "year", "month"], how="left")

n_missing = master["rainfall_mm"].isna().sum()
print(f"\nAfter CHIRPS join: {len(master):,} rows")
print(f"Missing rainfall: {n_missing} ({100*n_missing/len(master):.2f}%)")
print(f"Rainfall range: {master['rainfall_mm'].min():.1f} - {master['rainfall_mm'].max():.1f} mm")

CHIRPS: 30,384 rows × 7 cols
Columns: ['GID_3', 'NAME_3', 'NAME_2', 'NAME_1', 'year', 'month', 'rainfall_mm']

After CHIRPS join: 30,384 rows
Missing rainfall: 0 (0.00%)
Rainfall range: 0.0 - 417.2 mm


### 2c. Attach MODIS NDVI

Same pattern as CHIRPS. Every sector-month gets one NDVI value.

In [15]:
ndvi = pd.read_csv(NDVI_CSV)
print(f"NDVI: {len(ndvi):,} rows × {len(ndvi.columns)} cols")

ndvi_slim = ndvi[["GID_3", "year", "month", "ndvi"]]
master = master.merge(ndvi_slim, on=["GID_3", "year", "month"], how="left")

n_missing = master["ndvi"].isna().sum()
print(f"\nAfter NDVI join: {len(master):,} rows")
print(f"Missing NDVI: {n_missing} ({100*n_missing/len(master):.2f}%)")
print(f"NDVI range: {master['ndvi'].min():.3f} - {master['ndvi'].max():.3f}")

NDVI: 30,384 rows × 7 cols

After NDVI join: 30,384 rows
Missing NDVI: 33 (0.11%)
NDVI range: -0.013 - 0.904


### 2d. Attach DHS stunting (time-static broadcast)

DHS is observed only in 2019-20, but we need a stunting value for every sector-month. Strategy: broadcast the single 2019-20 value across all 72 months for each sector.

102 sectors have no DHS data and will have `stunting_rate = NaN` here that's intentional. Model training will filter to labeled rows.

In [16]:
# DHS_SECTOR path is declared in Cell 3 with the other path constants.
dhs_sector_df = pd.read_csv(DHS_SECTOR)
print(f"DHS sector data: {len(dhs_sector_df)} sectors")

# Keep only what we need (drop name columns — master already has them)
dhs_features = dhs_sector_df[[
    "GID_3", "n_clusters", "n_children", "stunting_rate", "severe_rate",
    "mean_haz", "mean_wealth_q", "mean_maternal_edu"
]].copy()

# Rename for clarity (these are DHS-derived baselines, not time-varying)
dhs_features = dhs_features.rename(columns={
    "n_clusters":        "dhs_n_clusters",
    "n_children":        "dhs_n_children",
    "stunting_rate":     "dhs_stunting_rate",
    "severe_rate":       "dhs_severe_rate",
    "mean_haz":          "dhs_mean_haz",
    "mean_wealth_q":     "dhs_wealth_q",
    "mean_maternal_edu": "dhs_maternal_edu",
})

# Left-join on GID_3 only (broadcasts the single DHS value across all months)
master = master.merge(dhs_features, on="GID_3", how="left")

n_labeled = master["dhs_stunting_rate"].notna().sum()
n_unlabeled = master["dhs_stunting_rate"].isna().sum()
print(f"\nAfter DHS join: {len(master):,} rows")
print(f"  Labeled (with DHS):    {n_labeled:>6,} = {n_labeled//72} sectors × 72 months")
print(f"  Unlabeled (no DHS):    {n_unlabeled:>6,} = {n_unlabeled//72} sectors × 72 months")
print(f"  Total sectors covered: {master.loc[master['dhs_stunting_rate'].notna(), 'GID_3'].nunique()} of 422")

DHS sector data: 320 sectors

After DHS join: 30,384 rows
  Labeled (with DHS):    23,040 = 320 sectors × 72 months
  Unlabeled (no DHS):     7,344 = 102 sectors × 72 months
  Total sectors covered: 320 of 422


### 2e. Attach WFP food prices with cascade fallback

WFP coverage is sparse (only 7 of 30 districts have direct data, ~6 commodities). To produce a price signal for every sector-month, we cascade:

1. **District match**: sector's district has WFP data → use district price
2. **Province fallback**: sector's province has any WFP-covered district → use province mean
3. **National fallback**: otherwise → use national monthly mean

WFP is also pivoted from long format (one row per commodity) to wide format (one column per commodity: `price_maize`, `price_beans`, etc.). The model decides which commodities matter.

In [17]:
wfp = pd.read_csv(WFP_CSV)
print(f"WFP raw: {len(wfp):,} rows × {len(wfp.columns)} cols")
print(f"  Districts: {wfp['admin2'].nunique()}, "
      f"commodities: {wfp['commodity'].nunique()}, "
      f"months: {wfp.groupby(['year','month']).ngroups}")

# Step 1: pivot WFP wide — one row per (admin1, admin2, year, month), one column per commodity
wfp_wide = wfp.pivot_table(
    index=["admin1", "admin2", "year", "month"],
    columns="commodity",
    values="price_rwf",
    aggfunc="mean",
).reset_index()

# Rename commodity columns to price_<commodity_slug>
def slug(name):
    return "price_" + (name.lower()
                       .replace(" ", "_")
                       .replace("(", "").replace(")", "")
                       .replace(",", ""))

commodity_cols = [c for c in wfp_wide.columns if c not in ["admin1", "admin2", "year", "month"]]
wfp_wide = wfp_wide.rename(columns={c: slug(c) for c in commodity_cols})
price_cols = [slug(c) for c in commodity_cols]

print(f"\nWFP pivoted wide: {len(wfp_wide):,} rows, {len(price_cols)} price columns:")
print(f"  {price_cols}")

WFP raw: 2,253 rows × 9 cols
  Districts: 7, commodities: 6, months: 72

WFP pivoted wide: 392 rows, 6 price columns:
  ['price_beans_dry', 'price_cassava_flour', 'price_maize', 'price_maize_flour', 'price_potatoes_irish', 'price_sorghum']


In [18]:
# Step 2: Province-month means (fallback level 2)
wfp_province = (
    wfp_wide.groupby(["admin1", "year", "month"])[price_cols]
    .mean()
    .reset_index()
    .rename(columns={c: c + "_prov_mean" for c in price_cols})
)

# Step 3: National-month means (fallback level 3)
wfp_national = (
    wfp_wide.groupby(["year", "month"])[price_cols]
    .mean()
    .reset_index()
    .rename(columns={c: c + "_nat_mean" for c in price_cols})
)

print(f"Province-month: {len(wfp_province):,} rows")
print(f"National-month: {len(wfp_national):,} rows")

Province-month: 248 rows
National-month: 72 rows


### 2f (interlude). Verify and map province name conventions

GADM uses Kinyarwanda province names (`NAME_1`) while WFP uses English (`admin1`). They will not join directly. We print both lists, declare a mapping, and apply it so the cascade fallback in the next cell works correctly.

If the assertion fails or a warning prints, adjust `province_kw_to_en` to match the WFP names exactly.

In [19]:
# Diagnostic: print both province conventions
gadm_provinces = sorted(master["NAME_1"].unique().tolist())
wfp_provinces  = sorted(wfp_wide["admin1"].unique().tolist())

print("GADM provinces (NAME_1):")
for p in gadm_provinces:
    print(f"  - {p}")
print("\nWFP provinces (admin1):")
for p in wfp_provinces:
    print(f"  - {p}")

# Mapping from GADM Kinyarwanda → WFP English province names.
# Verify the VALUES below match the WFP names printed above.
# Adjust spelling if WFP uses different conventions (e.g., "Northern" vs
# "Northern Province").
province_kw_to_en = {
    "Amajyaruguru":     "Northern Province",
    "Amajyepfo":        "Southern Province",
    "Iburasirazuba":    "Eastern Province",
    "Iburengerazuba":   "Western Province",
    "Umujyi wa Kigali": "Kigali City",
}

# Apply the mapping
master["province_en"] = master["NAME_1"].map(province_kw_to_en)

# Diagnostic 1: every GADM province must be in the mapping
unmapped = master[master["province_en"].isna()]["NAME_1"].unique()
if len(unmapped):
    print(f"\nERROR: GADM provinces not in mapping: {list(unmapped)}")
    print("       Update province_kw_to_en above and re-run.")
assert len(unmapped) == 0, "All GADM provinces must be mapped"

# Diagnostic 2: every mapped value should exist in the WFP data
mapped_values  = set(master["province_en"].unique())
wfp_values     = set(wfp_provinces)
missing_in_wfp = mapped_values - wfp_values

if missing_in_wfp:
    print(f"\nWARN: These mapped values do NOT appear in WFP data: {sorted(missing_in_wfp)}")
    print(f"      Available WFP names: {wfp_provinces}")
    print("      Adjust the mapping to use the exact WFP spelling, then re-run.")
else:
    print(f"\nAll {len(mapped_values)} mapped values are present in WFP data.")
    print("Mapping is correct, safe to continue.")

GADM provinces (NAME_1):
  - Amajyaruguru
  - Amajyepfo
  - Iburasirazuba
  - Iburengerazuba
  - Umujyi wa Kigali

WFP provinces (admin1):
  - Eastern Province
  - Kigali City
  - Northern Province
  - Southern Province
  - Western Province

All 5 mapped values are present in WFP data.
Mapping is correct, safe to continue.


In [20]:
# Step 4: Cascade joins — district → province → national
# WFP uses admin1=province (English), admin2=district
# We use province_en (mapped in the previous cell) to match WFP admin1

# Direct match: district price by (province_en, NAME_2, year, month)
master = master.merge(
    wfp_wide,
    left_on=["province_en", "NAME_2", "year", "month"],
    right_on=["admin1", "admin2", "year", "month"],
    how="left",
).drop(columns=["admin1", "admin2"])

# Province fallback
master = master.merge(
    wfp_province,
    left_on=["province_en", "year", "month"],
    right_on=["admin1", "year", "month"],
    how="left",
).drop(columns=["admin1"])

# National fallback
master = master.merge(wfp_national, on=["year", "month"], how="left")

# Cascade fill: for each commodity, prefer district → province → national
for c in price_cols:
    master[c] = master[c].fillna(master[c + "_prov_mean"])
    master[c] = master[c].fillna(master[c + "_nat_mean"])

# Drop the helper columns
master = master.drop(columns=[c + "_prov_mean" for c in price_cols] +
                           [c + "_nat_mean"  for c in price_cols])

# Diagnostics — show where each row's price came from
print("WFP price coverage after cascade fallback:")
for c in price_cols:
    n_miss = master[c].isna().sum()
    pct = 100 * (1 - n_miss / len(master))
    print(f"  {c:<30s} {pct:>6.2f}% coverage  ({n_miss:>5,} missing)")

WFP price coverage after cascade fallback:
  price_beans_dry                100.00% coverage  (    0 missing)
  price_cassava_flour             93.06% coverage  (2,110 missing)
  price_maize                    100.00% coverage  (    0 missing)
  price_maize_flour              100.00% coverage  (    0 missing)
  price_potatoes_irish            93.06% coverage  (2,110 missing)
  price_sorghum                   93.06% coverage  (2,110 missing)


### 2g. Master dataset summary

Final shape check before feature engineering.

In [21]:
# Final shape assertion: should be exactly 422 sectors × 72 months = 30,384 rows
expected_rows = 422 * 72
assert master.shape[0] == expected_rows, (
    f"Expected {expected_rows:,} rows but got {master.shape[0]:,} — "
    f"a merge probably created duplicates"
)

print(f"Master shape: {master.shape[0]:,} rows × {master.shape[1]} columns")
print(f"Memory:       {master.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print()
print("Columns and missingness:")
for c in master.columns:
    n_miss = master[c].isna().sum()
    pct_miss = 100 * n_miss / len(master)
    print(f"  {c:<35s} {pct_miss:>6.2f}% missing")

print()
print("Sample rows:")
print(master.head(3).to_string(index=False))

Master shape: 30,384 rows × 22 columns
Memory:       7.0 MB

Columns and missingness:
  GID_3                                 0.00% missing
  NAME_3                                0.00% missing
  NAME_2                                0.00% missing
  NAME_1                                0.00% missing
  year                                  0.00% missing
  month                                 0.00% missing
  rainfall_mm                           0.00% missing
  ndvi                                  0.11% missing
  dhs_n_clusters                       24.17% missing
  dhs_n_children                       24.17% missing
  dhs_stunting_rate                    24.17% missing
  dhs_severe_rate                      24.17% missing
  dhs_mean_haz                         24.17% missing
  dhs_wealth_q                         24.17% missing
  dhs_maternal_edu                     24.17% missing
  province_en                           0.00% missing
  price_beans_dry                       0.00% miss

In [22]:
# Save master dataset for downstream notebooks
OUTPUT_PATH = Path("../../data/processed/master_sector_month_raw.parquet")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
master.to_parquet(OUTPUT_PATH, index=False)

print(f"Saved: {OUTPUT_PATH.resolve()}")
print(f"{master.shape[0]:,} rows × {master.shape[1]} cols, "
      f"{OUTPUT_PATH.stat().st_size / 1e6:.2f} MB on disk")

Saved: C:\Users\Hello\kurinda\data\processed\master_sector_month_raw.parquet
30,384 rows × 22 cols, 0.69 MB on disk


## 3. Feature engineering

The raw master dataset has 22 columns of point-in-time signals. The model needs features that encode the **3–6 month etiological lag** between climate/price shocks and stunting outcomes. This block adds three feature families:

1. **Lag features**: rainfall, NDVI, and prices at t-1, t-3, t-6 months
2. **Climate anomalies**: z-scores against long-term sector means (deferred to Part 2)
3. **Cyclical seasonality**: sin/cos encoding of month (deferred to Part 2)

Every transformation is computed **within sector** (`groupby("GID_3")`) to respect the strong cross-sector heterogeneity in Rwanda's climate.

### 3a. Sort + reset index

Lag and rolling features require strict chronological ordering within each sector.

In [23]:
# Reload from parquet — defensive, in case the kernel was restarted
master = pd.read_parquet(Path("../../data/processed/master_sector_month_raw.parquet"))
print(f"Loaded master: {master.shape[0]:,} rows × {master.shape[1]} cols")

# Sort by (sector, year, month) and reset index — required for any lag operation
master = master.sort_values(["GID_3", "year", "month"]).reset_index(drop=True)

# Verify: every sector should have exactly 72 consecutive months
months_per_sector = master.groupby("GID_3").size()
assert (months_per_sector == 72).all(), (
    f"Some sectors don't have 72 months: "
    f"min={months_per_sector.min()}, max={months_per_sector.max()}"
)
print(f"Verified: all {master['GID_3'].nunique()} sectors have exactly 72 months each.")

Loaded master: 30,384 rows × 22 cols
Verified: all 422 sectors have exactly 72 months each.


### 3b. Impute time-series gaps (NDVI + WFP prices)

Two columns have residual missingness from upstream sources:

- `ndvi` (0.11% missing) — likely cloud-cover gaps in MODIS
- `price_cassava_flour`, `price_potatoes_irish`, `price_sorghum` (6.94% missing each) — months where no WFP district recorded the commodity, so even the national fallback couldn't fill it

We impute with **per-sector forward-fill then backward-fill**. This is the standard approach for short gaps in geophysical and economic time series, and it never leaks future information forward across sectors. After imputation we assert zero missing values.

In [24]:
# Columns that need temporal imputation (within sector)
impute_cols = [
    "ndvi",
    "price_beans_dry", "price_cassava_flour", "price_maize",
    "price_maize_flour", "price_potatoes_irish", "price_sorghum",
]

print("Before imputation:")
for c in impute_cols:
    pct = 100 * master[c].isna().sum() / len(master)
    print(f"  {c:<25s} {pct:>6.2f}% missing")

# Per-sector ffill then bfill (ffill alone leaves gaps at the start of a series)
master[impute_cols] = (
    master.groupby("GID_3")[impute_cols]
    .transform(lambda s: s.ffill().bfill())
)

print("\nAfter imputation:")
for c in impute_cols:
    pct = 100 * master[c].isna().sum() / len(master)
    print(f"  {c:<25s} {pct:>6.2f}% missing")

# Assert: every imputed column should now be 100% covered
remaining = {c: master[c].isna().sum() for c in impute_cols if master[c].isna().any()}
assert not remaining, f"Still missing after impute: {remaining}"
print("\nAll imputable columns are now 100% covered.")

Before imputation:
  ndvi                        0.11% missing
  price_beans_dry             0.00% missing
  price_cassava_flour         6.94% missing
  price_maize                 0.00% missing
  price_maize_flour           0.00% missing
  price_potatoes_irish        6.94% missing
  price_sorghum               6.94% missing

After imputation:
  ndvi                        0.00% missing
  price_beans_dry             0.00% missing
  price_cassava_flour         0.00% missing
  price_maize                 0.00% missing
  price_maize_flour           0.00% missing
  price_potatoes_irish        0.00% missing
  price_sorghum               0.00% missing

All imputable columns are now 100% covered.


### 3c. Lag features (1, 3, 6 months)

Chronic stunting is a **delayed-response outcome** — it reflects nutritional shocks that occurred 3–6 months earlier, during the critical window of in-utero and early-life growth. Current-month rainfall doesn't predict current stunting; *past* rainfall does.

For each time-varying signal, we create three lags:

- **t-1**: previous month (captures very recent shocks)
- **t-3**: quarter ago (captures the meaningful "season ago" signal)
- **t-6**: half year ago (matches the project's 3–6 month prediction horizon)

Sources lagged: `rainfall_mm`, `ndvi`, and all 6 price columns = 8 source signals × 3 lags = **24 new lag features**.

The first 6 months of each sector will have NaN lag values (no prior data to look back to). This is expected and will be handled at train/test split — those rows become unusable for lag-6 training but remain valid for shorter-lag features.

In [25]:
# Signals to lag (everything time-varying)
lag_sources = ["rainfall_mm", "ndvi"] + [c for c in master.columns if c.startswith("price_")]
lag_periods = [1, 3, 6]

print(f"Creating lag features for {len(lag_sources)} sources × {len(lag_periods)} periods = "
      f"{len(lag_sources) * len(lag_periods)} new columns")

# Group once, then shift within each sector
g = master.groupby("GID_3", sort=False)
for col in lag_sources:
    for lag in lag_periods:
        master[f"{col}_lag{lag}"] = g[col].shift(lag)

# Diagnostics — show NaN counts for lag features
lag_cols = [c for c in master.columns if "_lag" in c]
print(f"\nMaster shape after lags: {master.shape[0]:,} rows × {master.shape[1]} cols")
print(f"Total lag columns: {len(lag_cols)}")

# NaN count per lag period (expected: lag1 ≈ 422 NaN, lag3 ≈ 1,266, lag6 ≈ 2,532)
for lag in lag_periods:
    cols_at_this_lag = [c for c in lag_cols if c.endswith(f"_lag{lag}")]
    avg_nan = sum(master[c].isna().sum() for c in cols_at_this_lag) / len(cols_at_this_lag)
    expected = 422 * lag
    print(f"  lag{lag}: avg {avg_nan:.0f} NaN per column "
          f"(expected ~{expected:,} = 422 sectors × {lag} early months)")

Creating lag features for 8 sources × 3 periods = 24 new columns

Master shape after lags: 30,384 rows × 46 cols
Total lag columns: 24
  lag1: avg 422 NaN per column (expected ~422 = 422 sectors × 1 early months)
  lag3: avg 1266 NaN per column (expected ~1,266 = 422 sectors × 3 early months)
  lag6: avg 2532 NaN per column (expected ~2,532 = 422 sectors × 6 early months)


In [26]:
# Value-based leakage check: prove the lag columns reference PAST values, not future or current.
# This is a stronger test than the NaN-count check in Cell 46: it verifies the *direction*
# of the shift operation, not just its shape.

test_sector = master["GID_3"].iloc[0]
td = (
    master[master["GID_3"] == test_sector]
    .sort_values(["year", "month"])
    .reset_index(drop=True)
)

# Pick a row well past the max lag horizon so every lag has a valid lookup
i = 12
assert i >= max(lag_periods), "Test row must be past the max lag horizon"

print(f"Verifying lags on sector {test_sector}, row {i} "
      f"({td.loc[i, 'year']}-{td.loc[i, 'month']:02d}):\n")

# Positive check: every lag column equals the source column N rows earlier
for col in ["rainfall_mm", "ndvi", "price_maize"]:
    for lag in lag_periods:
        expected = td.loc[i - lag, col]
        observed = td.loc[i, f"{col}_lag{lag}"]
        match = abs(expected - observed) < 1e-6
        marker = "PASS" if match else "FAIL — LEAKAGE!"
        print(f"  {col}_lag{lag}: row {i-lag} value = {expected:.4f}, "
              f"lag column value = {observed:.4f}  [{marker}]")
        assert match, f"Lag value mismatch for {col}_lag{lag} — LEAKAGE!"

# Mutation check: lag values should NOT equal the current row's value
# (would indicate shift(0), i.e., no shift at all).
#
# Excluded from this check: imputed columns (prices). Cell 44's ffill→bfill can
# legitimately produce two adjacent rows with identical values, which would
# false-positive on this assertion. The positive check above already catches
# shift(0) bugs in essentially all realistic cases, so this is defense-in-depth
# rather than a primary safeguard. Continuous satellite signals (rainfall, NDVI)
# have zero chance of byte-identical adjacent months, so they're safe here.
for col in ["rainfall_mm", "ndvi"]:
    current = td.loc[i, col]
    lag1    = td.loc[i, f"{col}_lag1"]
    assert current != lag1, (
        f"{col}_lag1 equals current {col} — shift may not be applied at all"
    )

print("\nAll spot-checks PASSED. Lag features reference past values only "
      "(no leakage, no degenerate shift).")

Verifying lags on sector RWA.1.1.10_1, row 12 (2021-01):

  rainfall_mm_lag1: row 11 value = 142.3672, lag column value = 142.3672  [PASS]
  rainfall_mm_lag3: row 9 value = 202.9000, lag column value = 202.9000  [PASS]
  rainfall_mm_lag6: row 6 value = 5.5050, lag column value = 5.5050  [PASS]
  ndvi_lag1: row 11 value = 0.6616, lag column value = 0.6616  [PASS]
  ndvi_lag3: row 9 value = 0.6606, lag column value = 0.6606  [PASS]
  ndvi_lag6: row 6 value = 0.5033, lag column value = 0.5033  [PASS]
  price_maize_lag1: row 11 value = 300.0000, lag column value = 300.0000  [PASS]
  price_maize_lag3: row 9 value = 298.8900, lag column value = 298.8900  [PASS]
  price_maize_lag6: row 6 value = 309.3500, lag column value = 309.3500  [PASS]

All spot-checks PASSED. Lag features reference past values only (no leakage, no degenerate shift).


### 3d. Climate anomalies (sector-centered, pooled-scale z-score, leak-free)

Absolute rainfall and NDVI mix two signals: **where you are** (permanent geographic variation Rubavu is always wetter than Bugesera) and **what's happening now** (deviation from normal). For predicting risk shifts, only the second matters.

A natural first attempt is a per-sector z-score against the sector's own prior-year history for the same calendar month. This fails: with only 3-5 prior observations per (sector, month) group, the standard deviation collapses whenever a few priors happen to be close together, blowing up the resulting z-score by 100× or more. The robust alternative (median/MAD) has the same failure mode with 3 observations, one absolute deviation is always 0, so MAD reduces to the minimum gap between priors.

**My solution: keep the per-sector centering, but pool the scale estimate across sectors.**

anomaly[sector_X, March 2024] =

(value − mean(sector_X's prior Marches))

/ std(ALL sectors' Marches in years < 2024)

The centering still removes permanent location effects (Rubavu's wet baseline, Bugesera's dry baseline). The pooled std uses ~400+ observations per estimate (422 sectors × ≥1 prior year), so the scale is numerically stable regardless of how few years each individual sector contributes. The result is a properly bounded z-score (range ≈ ±3, no blow-ups) computed only from data strictly preceding each row's timestamp.

**Cost**: rows in 2020 have NaN anomalies (no prior data exists). All years 2021 onward are defined. That's 5 years of clean anomaly-enabled training data preserves most of the panel. LightGBM handles NaN natively as a learnable signal, so 2020 rows remain fully trainable on the 46 non-anomaly features.

**Empirical note**: rainfall anomalies show std ≈ 1.6 with heavy right tails, while NDVI anomalies show std ≈ 0.5 with near-Gaussian shape. This is real signal, not a defect: Rwandan rainfall is more volatile temporally (year-over-year at a single sector) than across sectors in any given month, while NDVI is smoother and behaves closer to a textbook z-score. Both features remain bounded within ±9 and feed cleanly into a tree-based model that is invariant to feature scaling.

In [27]:
# Climate anomalies — leak-free, sector-centered with pooled-scale z-score
#
# Why this formulation: short time-series (6 years per sector-month group) make
# any variance estimate computed from that single series structurally unstable.
# Both parametric std and robust MAD collapse to near-zero whenever 2-3 priors
# happen to be close, blowing up z-scores by 100x or more. We tested both and
# saw exactly this failure. The fix is to POOL the scale estimate across sectors:
#
#   - Centering: per-sector prior-year mean for the same calendar month
#                (removes permanent location effects, e.g. "Rubavu always wet")
#   - Scale:     std of ALL sectors' values for that calendar month, from years
#                strictly prior to the current row
#                (pooled n ~ 400+ per estimate — numerically rock-solid)
#
# Leak-free by construction: only year < current_year enters either calculation.

anomaly_sources = ["rainfall_mm", "ndvi"]
MIN_POOLED_OBS = 30  # pooled scale estimator's minimum sample size

for col in anomaly_sources:
    # Centering: per-sector × calendar-month, expanding mean of prior years
    sector_prior_mean = (
        master.groupby(["GID_3", "month"], observed=True)[col]
        .transform(lambda s: s.shift(1).expanding().mean())
    )

    # Scale: pooled cross-sector std from strictly prior years, per calendar month
    pooled_std = pd.Series(np.nan, index=master.index)
    for m in range(1, 13):
        month_rows = master[master["month"] == m]
        for y in sorted(month_rows["year"].unique()):
            prior_vals = master.loc[
                (master["month"] == m) & (master["year"] < y), col
            ]
            if len(prior_vals) >= MIN_POOLED_OBS:
                mask = (master["month"] == m) & (master["year"] == y)
                pooled_std.loc[mask] = prior_vals.std()

    master[f"{col}_anomaly"] = (master[col] - sector_prior_mean) / pooled_std

# Diagnostics
print(f"Anomaly features (sector-centered + pooled-scale z-score):\n")
for col in anomaly_sources:
    a = master[f"{col}_anomaly"]
    a_valid = a.dropna()
    n_null = a.isna().sum()
    pct_null = 100 * n_null / len(master)
    n_extreme = (a_valid.abs() > 3).sum()
    print(f"  {col}_anomaly:")
    print(f"    mean   = {a_valid.mean():+.4f}   (sector-centering keeps this near 0)")
    print(f"    std    = {a_valid.std():.4f}    (bounded — climate variability determines exact value)")
    print(f"    range  = [{a_valid.min():+.2f}, {a_valid.max():+.2f}]  (bounded within ~±9)")
    print(f"    |z|>3  = {n_extreme} of {len(a_valid):,} ({100*n_extreme/len(a_valid):.2f}%)")
    print(f"    NaN    = {n_null:,} ({pct_null:.1f}%) — year 2020 only")
    print()

print(f"Master shape after anomalies: {master.shape[0]:,} rows × {master.shape[1]} cols")

# Verify NaN pattern: only 2020 should be NaN
nan_by_year = master.groupby("year")["rainfall_mm_anomaly"].apply(lambda s: s.isna().mean() * 100)
print("\nNaN percentage by year:")
for year, pct in nan_by_year.items():
    yi = int(year)
    note = "(no prior data — anomaly undefined)" if yi == 2020 else "(anomaly defined from prior years)"
    print(f"  {yi}: {pct:5.1f}% NaN {note}")

Anomaly features (sector-centered + pooled-scale z-score):

  rainfall_mm_anomaly:
    mean   = -0.0094   (sector-centering keeps this near 0)
    std    = 1.6161    (bounded — climate variability determines exact value)
    range  = [-6.88, +8.26]  (bounded within ~±9)
    |z|>3  = 1690 of 25,320 (6.67%)
    NaN    = 5,064 (16.7%) — year 2020 only

  ndvi_anomaly:
    mean   = +0.0339   (sector-centering keeps this near 0)
    std    = 0.5225    (bounded — climate variability determines exact value)
    range  = [-7.20, +5.30]  (bounded within ~±9)
    |z|>3  = 9 of 25,320 (0.04%)
    NaN    = 5,064 (16.7%) — year 2020 only

Master shape after anomalies: 30,384 rows × 48 cols

NaN percentage by year:
  2020: 100.0% NaN (no prior data — anomaly undefined)
  2021:   0.0% NaN (anomaly defined from prior years)
  2022:   0.0% NaN (anomaly defined from prior years)
  2023:   0.0% NaN (anomaly defined from prior years)
  2024:   0.0% NaN (anomaly defined from prior years)
  2025:   0.0% NaN

### 3e. Cyclical seasonality (sin/cos encoding)

Month is an integer (1–12) but the model shouldn't treat it as a linear scale: December (12) and January (1) are seasonally adjacent, not 11 months apart. The standard fix is **cyclical encoding** with sine and cosine, which maps month onto a 2D unit circle:

month_sin = sin(2π · month / 12)

month_cos = cos(2π · month / 12)

This gives the model two continuous features where Dec and Jan are geometrically close in feature space. LightGBM can then split on these axes to learn seasonal patterns (e.g., "high stunting risk emerges after the long dry season" → cos < 0 region).

In [28]:
# Cyclical encoding of month
master["month_sin"] = np.sin(2 * np.pi * master["month"] / 12)
master["month_cos"] = np.cos(2 * np.pi * master["month"] / 12)

# Sanity check: every (sin, cos) point should lie on the unit circle
radii = np.sqrt(master["month_sin"]**2 + master["month_cos"]**2)
assert np.allclose(radii, 1.0), "month_sin/cos do not lie on unit circle"
print("Cyclical encoding verified — all (sin, cos) points lie on the unit circle.")

# Show one example per month so you can eyeball the geometry
print("\nMonth → (sin, cos) reference:")
for m in range(1, 13):
    row = master[master["month"] == m].iloc[0]
    print(f"  month {m:>2d} → sin={row['month_sin']:+.4f}, cos={row['month_cos']:+.4f}")

print(f"\nMaster shape after seasonality: {master.shape[0]:,} rows × {master.shape[1]} cols")

Cyclical encoding verified — all (sin, cos) points lie on the unit circle.

Month → (sin, cos) reference:
  month  1 → sin=+0.5000, cos=+0.8660
  month  2 → sin=+0.8660, cos=+0.5000
  month  3 → sin=+1.0000, cos=+0.0000
  month  4 → sin=+0.8660, cos=-0.5000
  month  5 → sin=+0.5000, cos=-0.8660
  month  6 → sin=+0.0000, cos=-1.0000
  month  7 → sin=-0.5000, cos=-0.8660
  month  8 → sin=-0.8660, cos=-0.5000
  month  9 → sin=-1.0000, cos=-0.0000
  month 10 → sin=-0.8660, cos=+0.5000
  month 11 → sin=-0.5000, cos=+0.8660
  month 12 → sin=-0.0000, cos=+1.0000

Master shape after seasonality: 30,384 rows × 50 cols


### 3f. Save engineered master dataset

`master_sector_month_features.parquet` is the input to notebook 03 (model training). It contains every feature the LightGBM will see:

- 4 admin/time keys (`GID_3`, `year`, `month`, `province_en`)
- 7 raw time-varying signals (rainfall, NDVI, 6 prices)
- 7 DHS baseline features (stunting_rate, severe_rate, wealth_q, etc.)
- 24 lag features (8 signals × 3 lags)
- 2 climate anomalies (leak-free)
- 2 seasonality features

We **do not drop** rows with NaN lag or anomaly values here. Those will be handled at the train/test split in notebook 03 — 2020–2021 rows are unusable for anomaly-enabled training but remain valid for lag-only models if needed.

In [29]:
# Final summary before save
print(f"Final master shape: {master.shape[0]:,} rows × {master.shape[1]} cols")
print(f"Memory:             {master.memory_usage(deep=True).sum() / 1e6:.1f} MB")

# Group columns by family for sanity
admin_cols     = ["GID_3", "NAME_3", "NAME_2", "NAME_1", "province_en", "year", "month"]
raw_signal     = ["rainfall_mm", "ndvi"] + [c for c in master.columns if c.startswith("price_") and "_lag" not in c]
dhs_cols       = [c for c in master.columns if c.startswith("dhs_")]
lag_cols       = [c for c in master.columns if "_lag" in c]
anomaly_cols   = [c for c in master.columns if c.endswith("_anomaly")]
season_cols    = ["month_sin", "month_cos"]

print(f"\nColumn families:")
print(f"  Admin/time:    {len(admin_cols):>3d} cols")
print(f"  Raw signals:   {len(raw_signal):>3d} cols")
print(f"  DHS baseline:  {len(dhs_cols):>3d} cols")
print(f"  Lag features:  {len(lag_cols):>3d} cols")
print(f"  Anomalies:     {len(anomaly_cols):>3d} cols")
print(f"  Seasonality:   {len(season_cols):>3d} cols")
total = len(admin_cols) + len(raw_signal) + len(dhs_cols) + len(lag_cols) + len(anomaly_cols) + len(season_cols)
print(f"  Total:         {total:>3d} cols  (master has {master.shape[1]})")

# Save
OUTPUT_PATH = Path("../../data/processed/master_sector_month_features.parquet")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
master.to_parquet(OUTPUT_PATH, index=False)

print(f"\nSaved: {OUTPUT_PATH.resolve()}")
print(f"  {master.shape[0]:,} rows × {master.shape[1]} cols, "
      f"{OUTPUT_PATH.stat().st_size / 1e6:.2f} MB on disk")

print("\nNotebook 02 complete. Next: 03_model.ipynb")
print("  - Define training target (binary stunting risk threshold)")
print("  - Time-based train/validation/test split")
print("  - Train LightGBM baseline, evaluate AUC-ROC, generate SHAP")

Final master shape: 30,384 rows × 50 cols
Memory:             13.8 MB

Column families:
  Admin/time:      7 cols
  Raw signals:     8 cols
  DHS baseline:    7 cols
  Lag features:   24 cols
  Anomalies:       2 cols
  Seasonality:     2 cols
  Total:          50 cols  (master has 50)

Saved: C:\Users\Hello\kurinda\data\processed\master_sector_month_features.parquet
  30,384 rows × 50 cols, 3.06 MB on disk

Notebook 02 complete. Next: 03_model.ipynb
  - Define training target (binary stunting risk threshold)
  - Time-based train/validation/test split
  - Train LightGBM baseline, evaluate AUC-ROC, generate SHAP


## Summary

Notebook 02 transforms five raw data sources into a 50-column feature matrix ready for supervised learning.

**Phase 1 — Spatial join**: 500 DHS clusters mapped to 320 of 422 GADM sectors (75.8% coverage). Weighted national stunting rate from the sector aggregate (33.4%) matches the published DHS 2019-20 figure (~33%), confirming the spatial join preserved the population structure.

**Phase 2 — Master dataset**: 30,384 sector-month observations (422 sectors × 72 months, 2020-2025) joining CHIRPS rainfall, MODIS NDVI, DHS baseline, and WFP food prices. Province name mismatch between GADM (Kinyarwanda) and WFP (English) was resolved with an explicit mapping. WFP prices use a district → province → national cascade fallback.

**Phase 3 — Feature engineering**: 24 lag features (8 sources × 3 horizons), 2 leak-free climate anomalies, 2 cyclical seasonality features. All temporal features verified leak-free either by value-based spot check (lags) or by construction (anomalies use strictly-prior-year baselines, seasonality has no temporal component).

**Output**: `data/processed/master_sector_month_features.parquet` — 30,384 rows × 50 cols, 3.06 MB on disk.

**Next**: notebook 03 trains a LightGBM baseline on this feature matrix.